In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Captures pour le rapport & la soutenance\n",
    "\n",
    "**Pipeline de données — Région Nouvelle-Aquitaine**\n",
    "\n",
    "Ce notebook affiche, étape par étape, les éléments à capturer (chaque cellule = une capture d'écran).\n",
    "Lance **« Run All »**, puis fais une capture de chaque cellule (code + résultat)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import glob\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "ROOT = \"c:/Users/ian chel/Desktop/MSPR - big data/economic-pulse-analyzer\"\n",
    "BRUT_ELEC  = f\"{ROOT}/MSPR_Final/MSPR/01_Donnees/brut/nouvelle_aquitaine_2012_2017_tour1.csv\"\n",
    "FUSION_DIR = f\"{ROOT}/MSPR_Final/MSPR/01_Donnees/delta_lake_final\"\n",
    "FINAL      = f\"{ROOT}/MSPR_Final/MSPR/01_Donnees/data_nouvelle_aquitaine_final.parquet\"\n",
    "pd.set_option(\"display.max_columns\", 12)\n",
    "print(\"Librairies et chemins chargés.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Données électorales brutes (avant transformation)\n",
    "Résultats des présidentielles 2012 + 2017 (1er tour) par canton, avant tout traitement."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "elec = pd.read_csv(BRUT_ELEC)\n",
    "print(f\"Dimensions : {elec.shape[0]} lignes x {elec.shape[1]} colonnes\")\n",
    "elec[[\"code_departement\", \"Libellé du département\", \"Libellé du canton\",\n",
    "      \"Inscrits\", \"Votants\", \"Exprimés\"]].head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Fusion élections + indicateurs (après fusion)\n",
    "Jointure des résultats électoraux et des indicateurs socio-économiques sur la clé géographique.\n",
    "\n",
    "```python\n",
    "# Code d'origine (notebook de préparation) :\n",
    "data_fusionnee = pd.merge(elec_df, df_indicateurs, on='CODGEO', how='inner')\n",
    "```"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fusion = pd.concat([pd.read_parquet(p) for p in glob.glob(f\"{FUSION_DIR}/*.parquet\")],\n",
    "                   ignore_index=True)\n",
    "print(f\"Dimensions après fusion : {fusion.shape[0]} lignes x {fusion.shape[1]} colonnes\")\n",
    "fusion[[\"Libellé du département\", \"Libellé du canton\", \"Nom\",\n",
    "        \"delta_P22_POP\", \"delta_P22_EMPLT\", \"delta_P22_POP1564\"]].head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Attributs socio-économiques utilisés\n",
    "Les variables `delta_` (variation des indicateurs entre deux périodes) qui alimentent les modèles."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "feature_cols = [c for c in fusion.columns\n",
    "                if c.startswith(\"delta_\") and \"pct\" not in c and \"eco\" not in c]\n",
    "print(f\"{len(feature_cols)} attributs delta_ utilisés :\\n\")\n",
    "for i, c in enumerate(feature_cols, 1):\n",
    "    print(f\"{i:2d}. {c}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Calcul des deltas (différence entre facteurs de même type)\n",
    "Chaque indicateur est transformé en **variation relative** entre deux périodes.\n",
    "\n",
    "```python\n",
    "# Code d'origine (notebook de préparation) :\n",
    "merged[f'delta_{col}'] = (merged[col_recent] - merged[col_old]) / (merged[col_old] + 1e-9)\n",
    "```"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "final = pd.read_parquet(FINAL)\n",
    "delta_view = [c for c in final.columns if c.startswith(\"delta_\")][:6]\n",
    "print(\"Statistiques des variables delta_ (base finale) :\")\n",
    "final[delta_view].describe().round(3).T[[\"mean\", \"std\", \"min\", \"50%\", \"max\"]]"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Nettoyage — qualité des données\n",
    "Vérification de l'absence de valeurs manquantes après traitement."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "manquantes = final.isnull().sum()\n",
    "print(f\"Total de valeurs manquantes : {int(manquantes.sum())}\")\n",
    "print(f\"Lignes : {final.shape[0]:,} | Colonnes : {final.shape[1]}\")\n",
    "manquantes[manquantes > 0] if manquantes.sum() else \"Aucune valeur manquante — base prête pour le Machine Learning.\""
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Base de données finale + corrélations\n",
    "Aperçu de la base finale et matrice de corrélation entre indicateurs socio-économiques."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "eco = [c for c in final.columns\n",
    "       if c.startswith(\"delta_\") and any(x in c.lower() for x in [\"pop\", \"emplt\", \"act\", \"log\", \"chom\"])][:9]\n",
    "display(final[eco].head(5))\n",
    "\n",
    "corr = final[eco].corr()\n",
    "plt.figure(figsize=(9, 7))\n",
    "sns.heatmap(corr, annot=True, fmt=\".2f\", cmap=\"coolwarm\", center=0,\n",
    "            linewidths=.5, cbar_kws={\"shrink\": .8})\n",
    "plt.title(\"Corrélations entre indicateurs socio-économiques (base finale)\")\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


NameError: name 'null' is not defined